# Inference Notebook (Today)\n\nLoad saved raw-stack artifacts and generate latest predictions for the trained universe.\n

In [ ]:
import os\nimport json\nimport pickle\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom dotenv import load_dotenv\n\nfrom modules.data.context import DataContext\nfrom modules.api import build_technical_dataframe, build_fundamental_dataframe, build_macro_dataframe\nfrom modules.engine.latest import score_latest_with_cluster_familiarity\nfrom modules.analysis import add_cluster_explanations\n\npd.set_option("display.max_colwidth", None)\n

In [ ]:
# ------------------------------------------------------------\n# 1) Load artifacts\n# ------------------------------------------------------------\nARTIFACT_DIR = Path("./artifacts/raw_stack")\nrequired = ["clf_raw.pkl", "reg_raw.pkl", "ae_raw.pkl", "flavor_space_raw.pkl", "meta.json"]\nmissing = [f for f in required if not (ARTIFACT_DIR / f).exists()]\nif missing:\n    raise FileNotFoundError(f"Missing artifacts in {ARTIFACT_DIR}: {missing}")\n\nwith open(ARTIFACT_DIR / "clf_raw.pkl", "rb") as f:\n    clf_raw = pickle.load(f)\nwith open(ARTIFACT_DIR / "reg_raw.pkl", "rb") as f:\n    reg_raw = pickle.load(f)\nwith open(ARTIFACT_DIR / "ae_raw.pkl", "rb") as f:\n    ae_raw = pickle.load(f)\nwith open(ARTIFACT_DIR / "flavor_space_raw.pkl", "rb") as f:\n    flavor_space = pickle.load(f)\nwith open(ARTIFACT_DIR / "meta.json", "r") as f:\n    meta = json.load(f)\n\nprint("Loaded artifacts from:", ARTIFACT_DIR.resolve())\nprint("meta keys:", sorted(meta.keys()))\n

In [ ]:
# ------------------------------------------------------------\n# 2) Build context + inference universe\n# ------------------------------------------------------------\nload_dotenv()\nFMP_API_KEY = os.getenv("FMP_API_KEY")\nif not FMP_API_KEY:\n    raise RuntimeError("Missing FMP_API_KEY in environment/.env")\n\nDATA_DIR = "./data"\nctx = DataContext.from_data_dir(\n    api_key=FMP_API_KEY,\n    data_dir=DATA_DIR,\n    db_name="quant.db",\n    sleep_s=0.0,\n    history_years=100,\n)\n\nif not isinstance(flavor_space.clustered_df.index, pd.MultiIndex) or "symbol" not in flavor_space.clustered_df.index.names:\n    raise RuntimeError("flavor_space.clustered_df does not contain (date, symbol) MultiIndex.")\n\nuniverse = sorted(set(flavor_space.clustered_df.index.get_level_values("symbol")))\nprint("Inference universe size:", len(universe))\n

In [ ]:
# ------------------------------------------------------------\n# 3) Build latest feature panel for inference\n# ------------------------------------------------------------\nend_date = pd.Timestamp.today().strftime("%Y-%m-%d")\nstart_date = (pd.Timestamp.today() - pd.DateOffset(years=3)).strftime("%Y-%m-%d")\n\ntechnical_df, technical_cols = build_technical_dataframe(\n    ctx=ctx,\n    symbols=universe,\n    start_date=start_date,\n    end_date=end_date,\n    verbose_data=False,\n)\n\nfund_df, fund_cols = build_fundamental_dataframe(\n    ctx=ctx,\n    symbols=universe,\n    start_date=start_date,\n    end_date=end_date,\n    target_index=technical_df.index,\n    daily_prices=technical_df,\n    verbose=False,\n)\n\nmacro_df, macro_cols = build_macro_dataframe(\n    ctx=ctx,\n    start_date=start_date,\n    end_date=end_date,\n    target_index=technical_df.index,\n    verbose=False,\n)\n\ninference_panel = pd.concat([technical_df, fund_df, macro_df], axis=1)\nprint("Inference panel shape:", inference_panel.shape)\n

In [ ]:
# ------------------------------------------------------------\n# 4) Score latest day + explanations\n# ------------------------------------------------------------\nlatest_date, scored = score_latest_with_cluster_familiarity(\n    train_data=inference_panel,\n    clf=clf_raw,\n    reg=reg_raw,\n    ae_model=ae_raw,\n    flavor_space=flavor_space,\n    market_position_value=0,\n    round_decimals=None,\n)\n\nscored = add_cluster_explanations(\n    scored,\n    flavor_space=flavor_space,\n    top_matches=5,\n    top_deviations=3,\n)\n\n# Optional: average-based blend\nscored["buy_score"] = (\n    pd.to_numeric(scored["clf__prob_buy"], errors="coerce").fillna(0.0)\n    + pd.to_numeric(scored["ranking"], errors="coerce").fillna(0.0)\n    + pd.to_numeric(scored["cluster_familiarity"], errors="coerce").fillna(0.0)\n) / 3.0\nscored["short_score"] = (\n    pd.to_numeric(scored["clf__prob_short"], errors="coerce").fillna(0.0)\n    + pd.to_numeric(scored["ranking"], errors="coerce").fillna(0.0)\n    + pd.to_numeric(scored["cluster_familiarity"], errors="coerce").fillna(0.0)\n) / 3.0\n\nprint(f"Latest date scored: {latest_date.date()} | rows={len(scored):,}")\n

In [ ]:
# ------------------------------------------------------------\n# 5) Filter + display predictions for today/latest\n# ------------------------------------------------------------\ncluster_col = str(flavor_space.cluster_col)\nmin_dist_col = f"{cluster_col}__min_dist"\n\nshow_cols = [\n    "clf__prob_buy",\n    "clf__prob_short",\n    "ranking",\n    cluster_col,\n    min_dist_col,\n    "cluster_familiarity",\n    "top_matching_features",\n    "top_deviating_features",\n    "cluster_mean_return",\n    "cluster_sharpe",\n    "cluster_mean_duration",\n    "buy_score",\n    "short_score",\n]\nshow_cols = [c for c in show_cols if c in scored.columns]\n\nbase_eligible = scored.loc[\n    (pd.to_numeric(scored["ranking"], errors="coerce") > 0.5)\n    & (pd.to_numeric(scored["cluster_familiarity"], errors="coerce") > 0.5)\n].copy()\n\nbuy_pool = base_eligible.loc[pd.to_numeric(base_eligible["clf__prob_buy"], errors="coerce") > 0.5].copy()\nshort_pool = base_eligible.loc[pd.to_numeric(base_eligible["clf__prob_short"], errors="coerce") > 0.5].copy()\n\nTOP_N_DISPLAY = 20\nbuy_top = buy_pool.nlargest(TOP_N_DISPLAY, "buy_score")[show_cols]\nshort_top = short_pool.nlargest(TOP_N_DISPLAY, "short_score")[show_cols]\n\nprint(\n    f"date={latest_date.date()} | rows={len(scored):,} "\n    f"| base_eligible={len(base_eligible):,} | buy_pool={len(buy_pool):,} | short_pool={len(short_pool):,}"\n)\ndisplay(buy_top)\ndisplay(short_top)\n